In [ ]:
# Imports and configuration
import os
import math
import requests
import json
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML
from urllib.parse import parse_qs, urlparse
import sys

# Backend base URL and fallback
API_BASE_URL = "http://localhost:3000/api/v1/lakehouse/network-analysis/transaction/"
FALLBACK_ACCOUNT_ID = "dbtrAcct_24a03dafa2c14f6da6bfc195d57c6d21"

def fetch_transaction_network(account_id):
    url = f"{API_BASE_URL}{account_id}"
    try:
        response = requests.get(url)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error fetching data: {e}")
        return None

def get_account_id_from_url():
    # Helper to get params in Voila
    if len(sys.argv) > 0 and '?' in sys.argv[0]:
        query = urlparse(sys.argv[0]).query
        params = parse_qs(query)
        return params.get('accountId', [FALLBACK_ACCOUNT_ID])[0]
    return os.getenv('ACCOUNT_ID', FALLBACK_ACCOUNT_ID)


In [ ]:
# Fetch Data
account_id = get_account_id_from_url()
data = fetch_transaction_network(account_id)


In [ ]:
if not data:
    display(HTML(f"<div style='color:#ef4444'>No network data available for account: {account_id}</div>"))
else:
    # Normalise Data
    nodes = []
    edges = []
    
    center = data.get('centerAccount')
    if center:
        nodes.append({
            'id': center.get('accountId'),
            'label': center.get('accountHolder'),
            'type': 'center',
            'status': 'normal', 
            'isCenter': True
        })
        
    for acc in data.get('connectedAccounts', []):
        nodes.append({
            'id': acc.get('accountId'),
            'label': acc.get('accountHolder'),
            'type': 'connected',
            'status': 'alert' if acc.get('hasAlert') else 'normal',
            'isCenter': False
        })
        
    edges = data.get('edges', [])
    
    # Simple Layout (Radial)
    positions = {}
    center_idx = 0
    # Find center index
    for i, n in enumerate(nodes):
        if n.get('isCenter'):
            center_idx = i
            break
            
    # Calculate positions
    radius = 300
    n_nodes = len(nodes)
    angle_step = 2 * math.pi / max(n_nodes - 1, 1)
    j = 0
    for i, node in enumerate(nodes):
        if i == center_idx:
            positions[node['id']] = (0.0, 0.0)
        else:
            angle = angle_step * j
            positions[node['id']] = (radius * math.cos(angle), radius * math.sin(angle))
            j += 1

    # Build Traces
    edge_x = []
    edge_y = []
    for e in edges:
        s = e.get('source')
        t = e.get('target')
        if s in positions and t in positions:
            x0, y0 = positions[s]
            x1, y1 = positions[t]
            edge_x += [x0, x1, None]
            edge_y += [y0, y1, None]

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y, 
        mode='lines', 
        line=dict(width=1, color='#cbd5e1', dash='solid'), 
        hoverinfo='none'
    )

    node_x = []
    node_y = []
    node_text = []
    marker_size = []
    marker_color = []
    node_ids = []

    for n in nodes:
        nid = n.get('id')
        if nid not in positions: continue
        x, y = positions[nid]
        node_x.append(x)
        node_y.append(y)
        label = n.get('label') or nid
        node_text.append(f"{label}")
        
        # Style logic
        is_center = n.get('isCenter')
        status = n.get('status')
        
        if is_center:
            color = '#f59e0b' # Amber center
            size = 32
        elif status == 'alert':
            color = '#ef4444' # Red alert
            size = 22
        else:
            color = '#6366f1' # Indigo normal
            size = 22
            
        marker_color.append(color)
        marker_size.append(size)
        node_ids.append(nid)

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode='markers+text',
        text=[t.split(' ')[0] for t in node_text], # Short label
        hovertext=node_text,
        hoverinfo='text',
        customdata=node_ids,
        marker=dict(
            showscale=False,
            color=marker_color,
            size=marker_size,
            line_width=2,
            line_color='#ffffff'
        ),
        textposition='bottom center'
    )

    fig = go.Figure(
        data=[edge_trace, node_trace],
        layout=go.Layout(
            title=None,
            showlegend=False,
            hovermode='closest',
            margin=dict(b=20,l=5,r=5,t=20),
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            height=600,
            paper_bgcolor='rgba(0,0,0,0)',
            plot_bgcolor='rgba(0,0,0,0)',
        )
    )

    # Render HTML
    graph_id = 'tx-network-graph'
    graph_html = pio.to_html(fig, full_html=False, config={'displayModeBar': False}, div_id=graph_id)
    
    # Sidebar Template (Light Theme)
    sidebar_html = """
    <div id="sidebar-container" style="width: 380px; background: #ffffff; color: #0f172a; padding: 24px; border-radius: 24px; box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.1); font-family: 'Inter', system-ui, sans-serif; margin-left: 24px; border: 1px solid #e2e8f0; display: flex; flex-direction: column; height: 600px; overflow-y: auto; transition: all 0.3s ease;">
        <!-- OVERVIEW STATE -->
        <div id="view-overview">
            <h2 style="margin: 0 0 20px 0; color: #0f172a; font-size: 1.25rem; font-weight: 700; border-bottom: 1px solid #e2e8f0; padding-bottom: 12px;">Transaction Network</h2>
            
            <div style="margin-bottom: 16px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Center Account</div>
                <div id="overview-id" style="font-weight: 600; font-size: 1rem; color: #0f172a;">-</div>
            </div>
            
            <div style="margin-bottom: 24px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Holder</div>
                <div id="overview-name" style="font-weight: 600; font-size: 1.1rem; color: #0284c7;">-</div>
            </div>
            
            <div style="background: #f8fafc; padding: 20px; border-radius: 20px; border: 1px solid #e2e8f0;">
                <h3 style="margin-top: 0; font-size: 0.85rem; color: #64748b; margin-bottom: 16px; text-transform: uppercase; font-weight: 600;">Summary</h3>
                <div style="display: grid; grid-template-columns: 1fr; gap: 16px;">
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Connected Accounts</span>
                        <span id="overview-connected" style="font-weight: 600; color: #0f172a;">0</span>
                    </div>
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Total Edges</span>
                        <span id="overview-edges" style="font-weight: 600; color: #0f172a;">0</span>
                    </div>
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Alerts</span>
                        <span id="overview-alerts" style="font-weight: 600; color: #dc2626;">0</span>
                    </div>
                </div>
            </div>
        </div>

        <!-- DETAILS STATE -->
        <div id="view-details" style="display: none;">
            <h2 style="margin: 0 0 20px 0; color: #0f172a; font-size: 1.25rem; font-weight: 700; border-bottom: 1px solid #e2e8f0; padding-bottom: 12px;">Node Details</h2>
            
            <div style="margin-bottom: 16px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Account ID</div>
                <div id="side-id" style="font-weight: 600; font-size: 1rem; color: #0f172a; word-break: break-all;">-</div>
            </div>
            
            <div style="margin-bottom: 16px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Holder</div>
                <div id="side-name" style="font-weight: 600; font-size: 1.1rem; color: #d97706;">-</div>
            </div>
            
            <div style="margin-bottom: 24px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Status</div>
                <div id="side-status" style="display: inline-block; padding: 4px 12px; border-radius: 9999px; font-size: 0.75rem; margin-top: 4px; font-weight: 500;">-</div>
            </div>

            <div style="background: #f8fafc; padding: 20px; border-radius: 20px; border: 1px solid #e2e8f0;">
                <h3 style="margin-top: 0; font-size: 0.85rem; color: #64748b; margin-bottom: 16px; text-transform: uppercase; font-weight: 600;">Connections</h3>
                <div style="display: grid; grid-template-columns: 1fr; gap: 16px;">
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Inbound</span>
                        <span id="side-inbound" style="font-weight: 600; color: #059669;">0</span>
                    </div>
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Outbound</span>
                        <span id="side-outbound" style="font-weight: 600; color: #0f172a;">0</span>
                    </div>
                </div>
            </div>
            
            <div style="margin-top: 24px; text-align: center;">
                <button onclick="resetSidebar()" style="background: white; border: 1px solid #cbd5e1; color: #64748b; padding: 8px 16px; border-radius: 8px; font-size: 0.75rem; cursor: pointer; font-weight: 500; transition: all 0.2s;">Back to Overview</button>
            </div>
        </div>
    </div>
    """
    
    # Client-side Logic
    interactive_script = f"""
    <script>
    (function() {{
        const nodes = {json.dumps(nodes)};
        const edges = {json.dumps(edges)};
        const centerAcct = {json.dumps(center)};
        const graphDiv = document.getElementById('{graph_id}');
        
        function initOverview() {{
            if(centerAcct) {{
                document.getElementById('overview-id').innerText = centerAcct.accountId || '-';
                document.getElementById('overview-name').innerText = centerAcct.accountHolder || '-';
            }}
            
            const connectedCount = nodes.filter(n => !n.isCenter).length;
            const alertCount = nodes.filter(n => n.status === 'alert').length;
            
            document.getElementById('overview-connected').innerText = connectedCount;
            document.getElementById('overview-edges').innerText = edges.length;
            document.getElementById('overview-alerts').innerText = alertCount;
        }}
        
        window.resetSidebar = function() {{
            document.getElementById('view-overview').style.display = 'block';
            document.getElementById('view-details').style.display = 'none';
            initOverview();
        }};
        
        function updateSidebar(nodeId) {{
            const node = nodes.find(n => n.id === nodeId);
            if (!node) return;
            
            document.getElementById('view-overview').style.display = 'none';
            document.getElementById('view-details').style.display = 'block';
            
            document.getElementById('side-id').innerText = node.id;
            document.getElementById('side-name').innerText = node.label || '-';
            
            const statusEl = document.getElementById('side-status');
            statusEl.innerText = node.status === 'alert' ? 'ALERT' : 'NORMAL';
            statusEl.style.background = node.status === 'alert' ? '#fef2f2' : '#f0f9ff';
            statusEl.style.color = node.status === 'alert' ? '#dc2626' : '#0369a1';
            
            let inbound = 0, outbound = 0;
            edges.forEach(e => {{
                if(e.source === nodeId) outbound++;
                if(e.target === nodeId) inbound++;
            }});
            
            document.getElementById('side-inbound').innerText = inbound;
            document.getElementById('side-outbound').innerText = outbound;
        }}
        
        setTimeout(initOverview, 500);
        
        if (graphDiv) {{
            graphDiv.on('plotly_click', d => {{
                if(d.points && d.points[0] && d.points[0].customdata) {{
                    updateSidebar(d.points[0].customdata);
                }}
            }});
            graphDiv.on('plotly_doubleclick', resetSidebar);
        }}
    }})();
    </script>
    """
    
    container_html = f"""
    <div style="display: flex; align-items: stretch; background-color: #ffffff; padding: 24px; border-radius: 32px; gap: 24px; border: 1px solid #f1f5f9;">
        <div style="flex: 1; background: #f8fafc; border-radius: 24px; border: 1px solid #e2e8f0; overflow: hidden; position: relative;">
            {graph_html}
        </div>
        {sidebar_html}
    </div>
    {interactive_script}
    """
    display(HTML(container_html))


In [ ]:
# Fetch transaction data for accountId from URL query param if present, else fallback
import os
from urllib.parse import parse_qs, urlparse
from IPython.display import display, HTML
import sys

def get_account_id_from_url():
    if 'VOILA_BASE_URL' in os.environ:
        if len(sys.argv) > 0 and '?' in sys.argv[0]:
            query = urlparse(sys.argv[0]).query
            params = parse_qs(query)
            return params.get('accountId', [FALLBACK_ACCOUNT_ID])[0]
    # Fallback: use fallback account id
    return FALLBACK_ACCOUNT_ID

account_id = get_account_id_from_url()
transaction_data, status_code = fetch_transaction_network(account_id)
# Only display the chart, no print or data dump


In [ ]:
import networkx as nx
import plotly.graph_objects as go

# Build the network graph from the API data
def build_network_graph(data):
    G = nx.DiGraph()
    # Add center node
    center = data['centerAccount']
    G.add_node(center['accountId'], label=center['accountHolder'], type='center', hasAlert=False)
    # Add connected accounts
    for acc in data.get('connectedAccounts', []):
        G.add_node(acc['accountId'], label=acc['accountHolder'], type='connected', hasAlert=acc['hasAlert'])
    # Add edges
    for edge in data.get('edges', []):
        G.add_edge(edge['source'], edge['target'], type=edge['type'])
    return G

if transaction_data:
    G = build_network_graph(transaction_data)
else:
    G = None

In [ ]:
import plotly.graph_objects as go
import networkx as nx
from IPython.display import display, HTML
import json

display(
    HTML(
        """
        <style>
            body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; margin: 0; padding: 0; background: #F9FAFB; }
            .legend-title { font-size: 18px; font-weight: 600; color: #111827; margin-bottom: 16px; }
            .legend-item { display: flex; align-items: center; gap: 12px; margin-bottom: 12px; font-size: 14px; color: #374151; }
            .legend-marker { width: 20px; height: 20px; border-radius: 50%; border: 2px solid #000; }
            .legend-marker.center { background: #F59E0B; border-width: 2px; }
            .legend-marker.normal { background: #6366F1; border-width: 2px; }
            .legend-marker.alert { background: #EF4444; border-width: 2px; }
            .legend-edge { width: 40px; height: 2px; background: #9CA3AF; border-top: 2px dashed #9CA3AF; }
            .section-title { font-size: 18px; font-weight: 600; color: #111827; margin-bottom: 24px; }
            .field { margin-bottom: 20px; }
            .field-label { font-size: 12px; font-weight: 500; color: #6B7280; text-transform: uppercase; margin-bottom: 4px; }
            .field-value { font-size: 14px; color: #111827; word-break: break-all; }
            .network-summary { margin-top: 32px; padding-top: 20px; border-top: 1px solid #E5E7EB; }
            .network-summary-title { font-size: 16px; font-weight: 600; color: #111827; margin-bottom: 16px; }
            .network-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 12px; font-size: 14px; color: #374151; }
        </style>
        """
    )
)


def _layout_center_middle(data):
    import math

    center = data['centerAccount']
    conn = data.get('connectedAccounts', [])
    pos = {center['accountId']: (0.5, 0.5)}
    n = len(conn)
    r = 0.35
    for i, acc in enumerate(conn):
        th = 2 * math.pi * i / n - math.pi / 2 if n else 0
        pos[acc['accountId']] = (0.5 + r * math.cos(th), 0.5 + r * math.sin(th))
    return pos


def _no_data_html():
    req = _last_request or {}
    err = _last_error or {}
    url = req.get('url', '')
    params = req.get('params', {})

    details = ""
    if err.get('type') == 'http':
        details = (
            f"<div style='margin-top:8px; color:#991B1B;'>HTTP {err.get('status','')} {err.get('reason','')}</div>"
            f"<pre style='margin-top:8px; padding:10px; background:#111827; color:#F9FAFB; border-radius:8px; overflow:auto; max-height:200px;'>{(err.get('body') or '')}</pre>"
        )
    elif err.get('type') == 'exception':
        details = f"<div style='margin-top:8px; color:#991B1B;'>Request failed: {err.get('message','')}</div>"

    return f"""
        <div style='padding: 12px; border: 1px solid #E5E7EB; border-radius: 8px; background: #F9FAFB;'>
          <div style='color:#6B7280;'>No network data to plot.</div>
          <div style='margin-top:8px; color:#111827; font-weight:600;'>Request</div>
          <div style='margin-top:4px; color:#374151; word-break: break-all;'><span style='color:#6B7280;'>URL:</span> {url}</div>
          <pre style='margin-top:8px; padding:10px; background:#FFFFFF; border:1px solid #E5E7EB; border-radius:8px; overflow:auto;'>{json.dumps(params, indent=2)}</pre>
          {details}
        </div>
    """


def plot_network(G, data):
    if G is None or len(G.nodes) == 0 or not data:
        display(HTML(_no_data_html()))
        return

    pos = _layout_center_middle(data)

    edge_x, edge_y = [], []
    for edge in G.edges(data=True):
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    node_x, node_y, node_text, node_color, node_border, customdata = [], [], [], [], [], []
    for node, nd in G.nodes(data=True):
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_text.append(f"{node}<br>{nd.get('label','')}")
        if nd['type'] == 'center':
            node_color.append('#F59E0B')
            node_border.append(10)
        elif nd['hasAlert']:
            node_color.append('#EF4444')
            node_border.append(6)
        else:
            node_color.append('#6366F1')
            node_border.append(4)
        customdata.append({
            'id': node,
            'label': nd.get('label', ''),
            'type': nd['type'],
            'hasAlert': nd.get('hasAlert', False),
        })

    edge_trace = go.Scatter(
        x=edge_x,
        y=edge_y,
        line=dict(width=2, color='#9CA3AF', dash='dash'),
        hoverinfo='none',
        mode='lines',
        showlegend=False,
    )
    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode='markers+text',
        marker=dict(
            size=[28] * len(node_x),
            color=node_color,
            line=dict(width=node_border, color='black'),
        ),
        text=node_text,
        textposition='bottom center',
        hoverinfo='text',
        customdata=customdata,
        showlegend=False,
    )

    fig = go.Figure(
        data=[edge_trace, node_trace],
        layout=go.Layout(
            showlegend=False,
            hovermode='closest',
            margin=dict(b=20, l=5, r=5, t=40),
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        ),
    )
    fig.update_layout(
        title='Transaction Network — center account in middle',
        height=500,
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
    )

    center_node = {
        'id': data['centerAccount']['accountId'],
        'label': data['centerAccount']['accountHolder'],
        'type': 'center',
        'hasAlert': False,
    }

    details_html = f"""
        <div id="details-container" style="display: flex; width: 100%; gap: 16px; margin-top: 16px;">
          <div style="width: 200px; background: #FFFFFF; border: 1px solid #E5E7EB; border-radius: 8px; padding: 24px; box-shadow: 0 1px 2px 0 rgba(0,0,0,0.05);">
            <div class="legend-title">Legend</div>
            <div class="legend-item"><div class="legend-marker center"></div>Center Account</div>
            <div class="legend-item"><div class="legend-marker normal"></div>Normal Account</div>
            <div class="legend-item"><div class="legend-marker alert"></div>Alert Triggered</div>
            <div class="legend-item"><div class="legend-edge"></div>Transaction Flow</div>
          </div>
          <div style="flex: 1; background: #FFFFFF; border: 1px solid #E5E7EB; border-radius: 8px; padding: 24px; box-shadow: 0 1px 2px 0 rgba(0,0,0,0.05);">
            <div class="section-title">Account Details</div>
            <div class="field">
              <div class="field-label">Account ID</div>
              <div class="field-value">{center_node['id']}</div>
            </div>
            <div class="field">
              <div class="field-label">Account Holder</div>
              <div class="field-value">{center_node['label']}</div>
            </div>
            <div class="field">
              <div class="field-label">Type</div>
              <div class="field-value">{'Center' if center_node['type'] == 'center' else 'Connected'}</div>
            </div>
            <div class="field">
              <div class="field-label">Alert Status</div>
              <div class="field-value">{'Alert Triggered' if center_node['hasAlert'] else 'Normal'}</div>
            </div>
            <div class="network-summary">
              <div class="network-summary-title">Network Summary</div>
              <div class="network-grid">
                <div><strong>Connected Accounts:</strong> 0</div>
                <div><strong>Outbound Connections:</strong> 0</div>
                <div><strong>Inbound Connections:</strong> 0</div>
                <div><strong>Accounts with Alerts:</strong> 0</div>
              </div>
            </div>
          </div>
        </div>
    """

    display(HTML(details_html))
    fig.show()
    display(
        HTML(
            f"""
            <script>
            window.networkData = {json.dumps(data)};
            function updateAccountDetails(nodeData) {{
                const container = document.getElementById('details-container');
                if (container && window.networkData) {{
                    const data = window.networkData;
                    const nodeId = nodeData.id;
                    const label = nodeData.label;
                    const nodeType = nodeData.type;
                    const hasAlert = nodeData.hasAlert;

                    let inbound = 0, outbound = 0, connected = 0, alerts = 0;
                    if (data && data.edges) {{
                        for (const e of data.edges) {{
                            if (e.target === nodeId) inbound++;
                            if (e.source === nodeId) outbound++;
                        }}
                    }}
                    if (data && data.connectedAccounts) {{
                        connected = data.connectedAccounts.filter(a => a.accountId === nodeId).length;
                        alerts = data.connectedAccounts.filter(a => a.accountId === nodeId && a.hasAlert).length;
                    }}

                    const rowHtml = `
                        <div style="width: 200px; background: #FFFFFF; border: 1px solid #E5E7EB; border-radius: 8px; padding: 24px; box-shadow: 0 1px 2px 0 rgba(0,0,0,0.05);">
                          <div class="legend-title">Legend</div>
                          <div class="legend-item"><div class="legend-marker center"></div>Center Account</div>
                          <div class="legend-item"><div class="legend-marker normal"></div>Normal Account</div>
                          <div class="legend-item"><div class="legend-marker alert"></div>Alert Triggered</div>
                          <div class="legend-item"><div class="legend-edge"></div>Transaction Flow</div>
                        </div>
                        <div style="flex: 1; background: #FFFFFF; border: 1px solid #E5E7EB; border-radius: 8px; padding: 24px; box-shadow: 0 1px 2px 0 rgba(0,0,0,0.05);">
                          <div class="section-title">Account Details</div>
                          <div class="field">
                            <div class="field-label">Account ID</div>
                            <div class="field-value">${{nodeId || ''}}</div>
                          </div>
                          <div class="field">
                            <div class="field-label">Account Holder</div>
                            <div class="field-value">${{label || ''}}</div>
                          </div>
                          <div class="field">
                            <div class="field-label">Type</div>
                            <div class="field-value">${{nodeType === 'center' ? 'Center' : 'Connected'}}</div>
                          </div>
                          <div class="field">
                            <div class="field-label">Alert Status</div>
                            <div class="field-value">${{hasAlert ? 'Alert Triggered' : 'Normal'}}</div>
                          </div>
                          <div class="network-summary">
                            <div class="network-summary-title">Network Summary</div>
                            <div class="network-grid">
                              <div><strong>Connected Accounts:</strong> ${{connected}}</div>
                              <div><strong>Outbound Connections:</strong> ${{outbound}}</div>
                              <div><strong>Inbound Connections:</strong> ${{inbound}}</div>
                              <div><strong>Accounts with Alerts:</strong> ${{alerts}}</div>
                            </div>
                          </div>
                        </div>
                    `;
                    container.innerHTML = rowHtml;
                }}
            }}
            document.addEventListener('DOMContentLoaded', function() {{
                setTimeout(function() {{
                    const gd = document.getElementsByClassName('plotly-graph-div')[0];
                    if (gd) {{
                        gd.on('plotly_click', function(data) {{
                            if (data.points && data.points.length) {{
                                const point = data.points[0];
                                if (point.customdata) {{
                                    updateAccountDetails(point.customdata);
                                }}
                            }}
                        }});
                    }}
                }}, 1000);
            }});
            </script>
            """
        )
    )


plot_network(G, transaction_data)